In [11]:
import torch
from torch import nn

# Base `nn.TransformerEncoderLayer` from PyTorch Documentation

Taken from here: https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html

In [12]:
batch_size = 3
seq_len = 10
embed_dim = 4
transformer_encoder_layer = nn.TransformerEncoderLayer(
    d_model=embed_dim, 
    nhead=2, 
    batch_first=True, 
    # src_key_padding_mask=a boolean tensor with True wherever there is padding token (it'll tell the encoder layer to apply ('-inf') to the dot product attention outputs wherever there are padding tokens)
)

# Claude Implementation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Example dataset with variable-length sequences
class VariableLengthDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx]

# Custom collate function
def collate_fn(batch):
    # batch is a list of tensors with shape [seq_len, embed_dim]
    lengths = torch.tensor([seq.shape[0] for seq in batch])
    
    # Pad sequences
    padded = pad_sequence(batch, batch_first=True, padding_value=0.0)
    # Shape: [batch_size, max_seq_len, embed_dim]
    
    # Create padding mask: True for padding positions, False for real tokens
    max_len = padded.shape[1]
    padding_mask = torch.arange(max_len).expand(len(batch), max_len) >= lengths.unsqueeze(1)
    # Shape: [batch_size, max_seq_len]
    
    return padded, padding_mask

# Create model
embed_dim = 128
num_heads = 8
encoder_layer = nn.TransformerEncoderLayer(
    d_model=embed_dim,
    nhead=num_heads,
    batch_first=True  # Important! Expects [batch, seq, feature]
)

# Create dummy data
sequences = [
    torch.randn(5, embed_dim),
    torch.randn(12, embed_dim),
    torch.randn(8, embed_dim),
    torch.randn(15, embed_dim),
]

dataset = VariableLengthDataset(sequences)
dataloader = DataLoader(dataset, batch_size=2, collate_fn=collate_fn)

# Training loop
for padded_batch, padding_mask in dataloader:
    # padded_batch: [batch_size, max_seq_len, embed_dim]
    # padding_mask: [batch_size, max_seq_len] - True where padding exists
    
    output = encoder_layer(padded_batch, src_key_padding_mask=padding_mask)
    # output: [batch_size, max_seq_len, embed_dim]
    
    print(f"Input shape: {padded_batch.shape}")
    print(f"Padding mask shape: {padding_mask.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Padding positions: {padding_mask}")
    print()

# Sebastian Raschka Implementation

Taken from here: https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/multihead-attention.ipynb

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head
        
        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [ ]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)